# Phase 2: Data Cleaning & Integrity Validation

**Project:** Customer Churn Prediction & Retention Analytics  
**Raw Source:** `data/raw/Telco_customer_churn.xlsx` (`Telco_Churn` sheet)  
**Cleaned Output:** `data/processed/telco_churn_clean.csv`  

### Objectives:
1. Load raw dataset and display baseline dimensions.
2. Strip leading/trailing whitespaces from column headers and string attributes.
3. Investigate the data type anomaly in `Total Charges` (object / whitespace strings).
4. Apply domain-defensible imputation for `Total Charges` based on customer tenure.
5. Check for duplicate records.
6. Validate numerical columns (`Tenure Months`, `Monthly Charges`, `Total Charges`) against impossible negative values.
7. Validate target integrity for `Churn Value` ({0, 1}).
8. Export the clean dataset to `data/processed/telco_churn_clean.csv`.

## 1. Import Libraries

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Set display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## 2. Load Raw Dataset & Inspect Shape

In [ ]:
raw_file = Path('../data/raw/Telco_customer_churn.xlsx')
df_raw = pd.read_excel(raw_file, sheet_name='Telco_Churn')

print(f"Raw Dataset Dimensions:")
print(f"- Rows: {df_raw.shape[0]:,}")
print(f"- Columns: {df_raw.shape[1]}")

## 3. Strip Whitespace from Column Headers & Text Fields

In [ ]:
df_clean = df_raw.copy()

# Strip whitespace from column headers
df_clean.columns = [col.strip() if isinstance(col, str) else col for col in df_clean.columns]

# Strip whitespace from string/object columns
for col in df_clean.columns:
    if df_clean[col].dtype == 'object' or isinstance(df_clean[col].dtype, pd.StringDtype):
        df_clean[col] = df_clean[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

print("Whitespace stripped successfully.")

## 4. Total Charges Investigation & Domain-Defensible Handling

In the raw dataset, `Total Charges` has an `object` data type due to whitespace string entries for certain rows.  
Let's coerce `Total Charges` to numeric and inspect all rows that become `NaN`.

In [ ]:
# Convert to numeric, turning invalid/whitespace strings into NaN
total_charges_numeric = pd.to_numeric(df_clean['Total Charges'], errors='coerce')
nan_mask = total_charges_numeric.isna()

print(f"Number of rows where Total Charges became NaN: {nan_mask.sum()}")
print("\nInspection of affected accounts:")
df_clean.loc[nan_mask, ['CustomerID', 'Tenure Months', 'Monthly Charges', 'Total Charges', 'Contract', 'Payment Method', 'Churn Value']]

### Why are these values missing?
- Exactly **11 customers** have missing/whitespace `Total Charges`.
- All 11 records have **`Tenure Months` = 0**.
- These are brand-new customers who signed up but have not yet completed their first monthly billing cycle.
- Consequently, their total accumulated billed charges are **$0.00**.
- Filling these with `0.0` is mathematically and domain-grounded, whereas filling with mean/median would distort new customer profiles.

In [ ]:
# Impute missing Total Charges with 0.0
df_clean['Total Charges'] = total_charges_numeric.fillna(0.0)
print(f"Total Charges dtype after conversion: {df_clean['Total Charges'].dtype}")
print(f"Remaining NaNs in Total Charges: {df_clean['Total Charges'].isna().sum()}")

## 5. Duplicate Rows Check

In [ ]:
dup_count = df_clean.duplicated().sum()
print(f"Duplicate records found: {dup_count}")

## 6. Numerical Range & Negative Values Validation

In [ ]:
num_cols = ['Tenure Months', 'Monthly Charges', 'Total Charges']
for col in num_cols:
    neg_count = (df_clean[col] < 0).sum()
    min_val = df_clean[col].min()
    max_val = df_clean[col].max()
    print(f"- {col}: Min = {min_val}, Max = {max_val}, Negative Values Count = {neg_count}")

## 7. Target Integrity Validation (`Churn Value`)

In [ ]:
unique_churn_values = df_clean['Churn Value'].unique()
print(f"Unique values in 'Churn Value': {unique_churn_values}")
assert set(unique_churn_values).issubset({0, 1}), "Target column contains invalid non-binary values!"

print("\nChurn Value Counts:")
print(df_clean['Churn Value'].value_counts(dropna=False))

## 8. Export Cleaned Dataset & Final Verification

In [ ]:
output_file = Path('../data/processed/telco_churn_clean.csv')
output_file.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(output_file, index=False)

print(f"Cleaned dataset saved to: {output_file.resolve()}")
print(f"Cleaned Dataset Shape: {df_clean.shape[0]:,} rows, {df_clean.shape[1]} columns.")

## 9. Data Cleaning Summary

| Check / Operation | Result |
|---|---|
| **Original Shape** | (7,043, 33) |
| **Final Shape** | (7,043, 33) |
| **Total Charges Issue** | 11 whitespace strings converted to `NaN` |
| **Total Charges Fix** | Imputed with `0.0` (all 11 accounts have `Tenure Months = 0`) |
| **Duplicates** | 0 duplicate records |
| **Negative Values** | 0 negative values across `Tenure Months`, `Monthly Charges`, `Total Charges` |
| **Target Validation** | `Churn Value` contains strictly binary values `{0, 1}` (5,174 retained, 1,869 churned) |
| **Output File** | `data/processed/telco_churn_clean.csv` |